# Question 1: What is a REST API?

REST API  is an approach to communicate software systems using over the internet standard HTTP methods. It is the most-used architectural style for designing web APIs.

In [ ]:
# Explain difference between:


# GET
- Does not modify anything on the server
- Parameters passed via URL query string

# POST
- Every call typically creates a new record
- Not idempotent — calling it twice creates two records

# PUT
- Sends a complete replacement of the resource
- Idempotent — calling it 10 times has the same result as calling it once

# DELETE
- Usually no request body
- Idempotent — deleting the same resource twice doesn't cause additional side effects

# Question 2: Create a FastAPI application with:
● Root endpoint (/)
● /courses GET endpoint
● /courses/{course_id} using path paramete

In [ ]:
from fastapi import FastAPI
app=FastAPI()
@app.get('/')
def root():
    return {'message':'Hello World'}

In [ ]:
@app.get("/", tags=["Root"])
def root():
    """Root endpoint — welcome message and App info."""
    return {
        "message": "Welcome to the Courses API 🎓",
        "version": "1.0.0",
        "endpoints": {
            "all_courses": "/courses",
            "single_course": "/courses/{course_id}",
            "docs": "/docs"
        }
    }


@app.get("/courses", response_model=List[Course], tags=["Courses"])
def get_courses():
    """Return a list of all available courses."""
    return courses_db


@app.get("/courses/{course_id}", response_model=Course, tags=["Courses"])
def get_course(course_id: int):
    """Return a single course by its ID."""
    course = next((c for c in courses_db if c.id == course_id), None)
    if not course:
        raise HTTPException(
            status_code=404,
            detail=f"Course with id {course_id} not found."
        )
    return course


#Question 3:Add validation to:
- course_id must be integer > 0
- level must be one of: beginner, intermediate, advanced
#What happens if validation fails?

In [ ]:
1. course_id must be integer > 0

Uses FastAPI's Path() with gt=0 (greater than 0):

pythoncourse_id: int = Path(..., gt=0)

/courses/3 ✅

POST: /courses/0 ❌ → 422 Unprocessable Entity

/courses/-5 ❌ → 422 Unprocessable Entity

/courses/abc ❌ → 422 Unprocessable Entity

2. level must be beginner or intermediate or advanced

it uses a str Enum named Level:

pythonclass Level(str, Enum):

beginner = "beginner"

intermediate = "intermediate"

advanced = "advanced"

Not applied to the Course model so feature is valid over stored data

This is also used as an optional query filter on GET /courses:

/courses? level=beginner ✅ → returns beginner courses only

/courses? level=expert ❌ → 422 Unprocessable Entity

SyntaxError: unterminated string literal (detected at line 3) (4153536369.py, line 3)

# Question 5: Explain:
- Authentication vs Authorization
- JWT-based authentication
- Why stateless authentication is scalable

In [ ]:
  #Authentication vs Authorization
# Example of an identification:
#Authentication = showing your ID at check-in to prove who you are
#Authorization = your room key only opens your room, not others


# JWT-Based Authentication
- JWT (JSON Web Token) is a compact, self-contained token used to securely transmit identity information between a client and server.

# Why stateless authentication is scalable
Traditional (stateful) sessions store session data on the server:
User logs in → Server stores session in memory/DB
                → Every request must hit the same server
                   (or share a central session store)

# Question 6: Why is API testing critical?\
Difference between:
- Unit testing
- Integration testing

In [ ]:
#APIs are the heart of the modern application: they glue front-ends, mobile apps, third-party services and microservices together. If a part of the API, API is broken then it breaks everything that depends on it.

- Unite testing
#Ans its very fast , single funtion, scope narrow
- integration testion
#Ans multipule componants together, slower and scope broad

In [ ]:
#Write a sample test case using FastAPI TestClient.

In [ ]:
from fastapi import FastAPI
app=FastAPI()
@app.get('/')
def root():
    return {'message':'Hello World'}

In [ ]:
#Question 7:Explain use of:


In [ ]:
# 1. Optional
# Marks a field or parameter as either a value or None — it may or may not be present.
from typing import Optional

# Without Optional — level is always required
def get_courses(level: str):
    ...

# With Optional — level can be a string OR None (default)
def get_courses(level: Optional[str] = None):
    if level:
        return [c for c in courses_db if c.level == level]
    return courses_db
# Optional[str] is just shorthand for Union[str, None].
# Use it whenever a parameter or field is not mandatory.

# 2. List[str]
# Specifies a list where every item must be a specific type.
from typing import List

# A plain list — no info about what's inside
def get_tags(tags: list):
    ...

# A typed list — clearly a list of strings
def get_tags(tags: List[str]):
    ...

# Used in Pydantic models too
# class Course(BaseModel):
#     title: str
#     tags: List[str]          # e.g. ["python", "beginner"]
#     prerequisites: List[int] # e.g. [1, 2] (course IDs)
# Prevents mixing types inside a list and makes the shape of data immediately obvious.

# 3. Union
# Allows a value to be one of several types.
from typing import Union

# Accepts either an int or a string ID
def find_course(course_id: Union[int, str]):
    ...

# A field that can be a number or a descriptive string
# class Course(BaseModel):
#     duration: Union[int, str]  # e.g. 10 (hours) or "self-paced"


# 4. Annotated
# Attaches extra metadata (validation rules, descriptions) to a type — keeps type and constraints together in one place.
from typing import Annotated
from fastapi import Path, Query

# Without Annotated — constraints mixed into function signature
def get_course(course_id: int = Path(..., gt=0, description="Must be > 0")):
    ...

# With Annotated — type and rules bundled cleanly
# CourseId = Annotated[int, Path(..., gt=0, description="Must be > 0")]

# def get_course(course_id: CourseId):
#     ...

# Reusable across multiple endpoints
# def update_course(course_id: CourseId):
#     ...

# def delete_course(course_id: CourseId):
#     ...
# Annotated shines when you have reusable validation rules — define once, apply everywhere.

# Side-by-Side Summary
# ToolPurposeExampleOptional[X]Value may be NoneOptional[str] = NoneList[X]A list of typed itemsList[str]Union[X, Y]One of multiple typesUnion[int, str]Annotated[X, ...]Type + metadata/validationAnnotated[int, Path(gt=0)]

# Why Proper Typing Improves Maintainability
# 1. Self-documenting code
# Types tell the next developer (or future you) exactly what a function expects and returns — no need to read the full implementation:
# # Unclear — what is data? what does it return?
# def process(data):
#     ...


# def process(courses: List[Course]) -> Optional[Course]:
#     ...
# 2. Catches bugs before runtime
# Type checkers like mypy or IDE linters flag mismatches at write time, not in production:
# get_course(course_id="abc")  # mypy flags this immediately — expected int
# 3. FastAPI generates validation + docs automatically
# Types drive both request validation and the /docs Swagger UI — no extra work needed:
# # FastAPI reads this type and auto-validates + documents it
# course_id: Annotated[int, Path(gt=0)]
# 4. Easier refactoring
# When types are explicit, IDEs can safely rename, trace usages, and detect breaking changes across the entire codebase.
# 5. Better team collaboration
# A typed API is a contract — frontend, backend, and QA teams all know exactly what shape the data takes without back-and-forth communication.

# Question 8:Explain the difference between:
- Path parameter -
Embedded directly in the URL path, typically to identify a specific resource.

# When should each be used?
- Identifying a specific resource by its ID
-When the value is required and makes no sense without it
- Navigating  through a resource hierarchy

- Query Parameter
Appended to the URL after a ?, as key-value pairs separated by &.

# When should each be used?
- Filtering, sorting, or paginating a collection
- Optional parameters that refine a request
- Search operations where multiple combinations are possible

- Request Body- Data sent in the body of the HTTP request, typically as JSON

# When should each be used?
- Creating or updating a resource
- Sending complex or large amounts of data
- Sending sensitive data (tokens) — not exposed in the URL

# Question 9:
Difference between:
-  BaseModel
- Field
- validator
- root_validator

In [3]:
#1. basemodel-
 #When an object is created, Pydantic automatically validates data types.
#What it provides you:

#Automatic type validation at instantiation Utility methods .dict(), .json(), .copy()
#Clear error messages for invalid data entry
#2. Field Used to add additional rules and metadata to individual fields – beyond just the type hint.
#3.validator
#A custom function to validate or transform a single field value after type checking.
# 4.root_validator
#Like validator, but runs on the entire model's data at once — used when validation logic involves multiple fields together.